# Part 5 · Agent Substrate: your agents as actors in gVisor sandboxes

Parts 1-4 secured *service* and *agent* traffic. This part is about **where an agent runs**. With **Agent Substrate**, kagent runs each agent as an **actor**: a gVisor-sandboxed process that lives on a pool of pre-warmed **workers** while it is serving a turn, and is a **snapshot in object storage** the rest of the time. You deploy a `SandboxAgent` instead of an ordinary pod, and kagent does the rest: one golden snapshot per agent, one actor per conversation, resume on request, suspend after the reply.

Agent Substrate is an open-source project ([agent-substrate/substrate](https://github.com/agent-substrate/substrate), Apache-2.0) and kagent is the control plane on top of it: the CRD, the sessions, the tools, the A2A endpoint, the inventory API, and the rollout of a change to the agent's shape as a new golden snapshot beside the old one.

> **Beta / off by default.** Substrate ships in kagent-enterprise ≥ v0.5.2 and is disabled until you turn it on. It runs on kind with **gVisor (runsc)**, no `/dev/kvm` needed, because gVisor is a userspace kernel.

Start with the **Connect** cell at the bottom (and **Enable substrate** once per cluster). Every cell above the line is then one step of the demo.

1. Prove the agent runs in a gVisor sandbox
2. One turn, as kagent sees it
3. Concurrency is the worker count
4. Compare actor and pod resource use
5. Golden actor and snapshot resume
6. Tools from inside the sandbox
7. A second pool for a different trust tier
8. Change the agent's shape: a second golden beside the first
9. Pod agents and sandboxed agents in one graph
10. The worker fleet
11. Chat with the sandboxed agent
12. Watch it live: Substrate Scope


### The substrate components

`substrate-up.sh` turns on kagent's **Agent Substrate** engine (API group `ate.dev`; every component is prefixed `ate*`). What it adds to the `kagent` namespace:

| Component | Role |
|---|---|
| **SandboxAgent** (`kagent.dev`) | the agent you deploy: same spec as `Agent` (model, system message, tools, memory) plus a `substrate` block; runs as a gVisor actor, not a pod |
| **WorkerPool** (`ate.dev`) | a pool of pre-warmed worker pods (`ateom-gvisor` image); one running actor per worker at a time, any number suspended |
| **ActorTemplate** (`ate.dev`) | what kagent renders from a SandboxAgent: image, env, readiness, snapshot location; each shape gets its own golden snapshot |
| **SandboxConfig** (`ate.dev`) | the `runsc` binary per architecture, pinned by sha256 |
| **ate-api-server** | the control plane: actors, workers, placement, resume and suspend (gRPC, `kagent-api.kagent.svc:443`) |
| **ate-controller** | reconciles `WorkerPool`s into worker Deployments and builds the golden actors |
| **atelet** (DaemonSet) | per node: runs checkpoint and restore for the workers on that node, moves snapshots to and from object storage |
| **atenet-router** | routes a turn to whichever worker holds the actor, resuming it first if it is suspended; the data plane is agentgateway |
| **rustfs** | the in-cluster S3-compatible bucket the snapshots land in (`ate-snapshots`) |
| **valkey** | actor and worker state store |

The kagent controller talks to `ate-api-server` and exposes the whole inventory on one REST call, `GET /api/substrate/status`, which several cells below read.


## 5.1 · Step 1: prove the agent runs in a gVisor sandbox

<div align="center"><svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 720 260" style="width:100%;max-width:1000px;height:auto" font-family="-apple-system,Segoe UI,Roboto,sans-serif"><rect x="0" y="0" width="720" height="260" rx="10" fill="#f8fafc"/><text x="360" y="24" text-anchor="middle" font-size="14.5" font-weight="700" fill="#0f172a">Step 1 · Where your agent actually runs: a gVisor-sandboxed actor</text><defs><marker id="g" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#16a34a"/></marker><marker id="d" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#d97706"/></marker><marker id="n" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#334155"/></marker><marker id="v" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#8b5cf6"/></marker></defs><rect x="14" y="66" width="116" height="46" rx="8" fill="#e0e7ff" stroke="#6366f1" stroke-width="1.5"/><text x="72" y="86" text-anchor="middle" font-size="9.5" font-weight="700" fill="#312e81">SandboxAgent</text><text x="72" y="101" text-anchor="middle" font-size="8" fill="#4338ca">(kagent CRD)</text><line x1="130" y1="89" x2="168" y2="89" stroke="#334155" stroke-width="1.7" marker-end="url(#n)"/><rect x="170" y="66" width="120" height="46" rx="8" fill="#dbeafe" stroke="#60a5fa" stroke-width="1.4"/><text x="230" y="86" text-anchor="middle" font-size="9.5" font-weight="700" fill="#1e293b">kagent-</text><text x="230" y="101" text-anchor="middle" font-size="9.5" font-weight="700" fill="#1e293b">controller</text><line x1="290" y1="89" x2="330" y2="89" stroke="#334155" stroke-width="1.7" marker-end="url(#n)"/><text x="310" y="80" text-anchor="middle" font-size="7.5" fill="#475569">ActorTemplate</text><rect x="332" y="66" width="110" height="46" rx="8" fill="#dbeafe" stroke="#60a5fa" stroke-width="1.4"/><text x="387" y="86" text-anchor="middle" font-size="9.5" font-weight="700" fill="#1e293b">ate-api-</text><text x="387" y="101" text-anchor="middle" font-size="9.5" font-weight="700" fill="#1e293b">server</text><line x1="442" y1="89" x2="468" y2="89" stroke="#334155" stroke-width="1.7" marker-end="url(#n)"/><text x="455" y="80" text-anchor="middle" font-size="7.5" fill="#475569">bind</text><rect x="470" y="52" width="236" height="120" rx="8" fill="#dcfce7" stroke="#16a34a" stroke-width="1.6"/><text x="588" y="70" text-anchor="middle" font-size="9" font-weight="700" fill="#14532d">WorkerPool worker · ateom-gvisor</text><rect x="486" y="84" width="204" height="74" rx="8" fill="#fef3c7" stroke="#d97706" stroke-width="1.6"/><text x="588" y="102" text-anchor="middle" font-size="9" font-weight="700" fill="#7c2d12">gVisor sandbox (runsc)</text><text x="588" y="120" text-anchor="middle" font-size="10" font-weight="700" fill="#92400e">ADK actor</text><text x="588" y="138" text-anchor="middle" font-size="8" fill="#b45309">guest kernel ≠ host kernel</text><rect x="300" y="196" width="220" height="40" rx="8" fill="#ede9fe" stroke="#8b5cf6" stroke-width="1.4"/><text x="410" y="214" text-anchor="middle" font-size="9" font-weight="700" fill="#5b21b6">kagent-atelet (DaemonSet)</text><text x="410" y="228" text-anchor="middle" font-size="8" fill="#6d28d9">installs runsc on the node</text><line x1="520" y1="206" x2="588" y2="172" stroke="#8b5cf6" stroke-width="1.7" stroke-dasharray="5 3" marker-end="url(#v)"/><text x="566" y="192" text-anchor="middle" font-size="7.5" fill="#6d28d9">runsc</text><text x="360" y="252" text-anchor="middle" font-size="10" fill="#64748b">A SandboxAgent → ActorTemplate → bound onto a pooled gVisor worker; the agent runs in a runsc sandbox with its own guest kernel.</text></svg></div>

In [ ]:
show $SUBSTRATE_YAML/substrate-demo.yaml
kubectl --context $CTX apply -f $SUBSTRATE_YAML/substrate-demo.yaml
wait-ready sandboxagent substrate-demo $SUBSTRATE_YAML/substrate-demo.yaml


In [ ]:
# the declarations: a gVisor worker pool, and the ActorTemplate kagent rendered for the agent
kubectl --context $CTX -n kagent get workerpool kagent-default -o custom-columns='POOL:.metadata.name,CLASS:.spec.sandboxClass,REPLICAS:.spec.replicas,IMAGE:.spec.ateomImage'
templates substrate-demo


### Catch gVisor in the act

The declarations above say `gvisor`. This proves it. An idle actor is a **snapshot on disk with no
process**, so `runsc` exists only while a turn is being served: fire a request and watch the node
during it. kind runs each Kubernetes node as a container, so we can look straight at the node's
process table.

In [ ]:
catch-runsc substrate-demo "Write a short paragraph about sandboxes."


### At rest an actor costs nothing, and every session gets its own

Now that the turn is over, look again. The session's actor was suspended straight after the reply: it is a snapshot, not a process. The worker keeps one **golden** actor per `SandboxAgent` and one **actor per session**, which is the isolation boundary that matters for multi-tenancy: two conversations with the same agent are two separate gVisor sandboxes. Below, kagent's inventory API is the source of truth for each actor's state, and the node's process table is cross-checked against it.


In [ ]:
ask substrate-demo "Say hello in five words."     # a SECOND conversation with the same agent
sleep 6                                            # let the actor suspend
actors substrate-demo                              # kagent's inventory: the golden actor and one actor per session
on-node substrate-demo                             # the same actors on the worker node: no runsc, snapshots only


## 5.2 · One turn, as kagent sees it

kagent joins the substrate inventory to its own sessions on `GET /api/substrate/status`: every WorkerPool, every ActorTemplate with its golden snapshot, every actor with its state and snapshot version, and every worker with the pod behind it. Fire one turn and poll that endpoint while it runs. The actor goes **Suspended → Resuming → Running → Suspending → Suspended**, and its snapshot version goes up by one, because kagent suspends the actor the moment the reply has been read. There is no idle timer to tune: the worker is free for the next conversation as soon as this one has its answer.


In [ ]:
watch-turn substrate-demo "Write two sentences about checkpoint and restore."


## 5.3 · Concurrency is the worker count

A worker runs **one actor at a time**, and it takes any number of suspended ones. So the pool size is the number of turns that can be served at once, not the number of agents you can have. Below, three conversations fire at the same agent on a two-worker pool at the same moment: two get a worker and run, the third is refused straight away with the remedy in the message (*no free workers; try again later or increase WorkerPool replicas*). Then the pool is scaled to three with a plain `kubectl scale`, because `WorkerPool` has a scale subresource, and the same three run together.


In [ ]:
fire 3 substrate-demo        # three turns at the same moment, two workers
scale-pool 3                 # kubectl scale workerpool/kagent-default --replicas=3, and wait for the store
fire 3 substrate-demo        # the same three turns, three workers
scale-pool 2


## 5.4 · Compare actor and pod resource use

Two properties matter here: standing up another agent does **not** cost another pod, and the
workload really is sandboxed. Below, three more `SandboxAgent`s bind onto the workers already
running, timed to the millisecond, and then the **same three agents as ordinary `Agent`s** for
comparison, which is where the pods and the reserved memory show up. Many isolated actors, few pods.

In [ ]:
kubectl --context $CTX -n kagent get pods -l ate.dev/worker-pool=kagent-default --no-headers | wc -l | tr -d ' ' | sed 's/^/worker pods before: /'
for n in 2 3 4; do time-ready sandboxagent substrate-demo-$n $SUBSTRATE_YAML/density-sandboxagents.yaml; done
kubectl --context $CTX -n kagent get pods -l ate.dev/worker-pool=kagent-default --no-headers | wc -l | tr -d ' ' | sed 's/^/worker pods after:  /'
kubectl --context $CTX -n kagent get actortemplate --no-headers | wc -l | tr -d ' ' | sed 's/^/gVisor actors (templates): /'


Now the alternative: the **same three agents as ordinary `Agent`s**, each in its own pod, which is where the Deployments and the reserved memory show up.


In [ ]:
kubectl --context $CTX apply -f $SUBSTRATE_YAML/density-pod-agents.yaml
for n in 1 2 3; do kubectl --context $CTX -n kagent wait agent/pod-baseline-$n --for=condition=Ready --timeout=180s >/dev/null; done
kubectl --context $CTX -n kagent get pods -l app.kubernetes.io/managed-by=kagent -o custom-columns='POD:.metadata.name,MEM_REQUEST:.spec.containers[0].resources.requests.memory,MEM_LIMIT:.spec.containers[0].resources.limits.memory' | grep -E 'POD|pod-baseline'
echo "   3 SandboxAgents -> 0 extra pods.  3 Agents -> 3 pods and their memory, reserved while idle."
kubectl --context $CTX -n kagent delete sandboxagent substrate-demo-2 substrate-demo-3 substrate-demo-4 --ignore-not-found --wait=false >/dev/null
kubectl --context $CTX -n kagent delete agent pod-baseline-1 pod-baseline-2 pod-baseline-3 --ignore-not-found --wait=false >/dev/null


## 5.5 · Golden actor and snapshot resume

<div align="center"><svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 720 232" style="width:100%;max-width:1000px;height:auto" font-family="-apple-system,Segoe UI,Roboto,sans-serif"><rect x="0" y="0" width="720" height="232" rx="10" fill="#f8fafc"/><text x="360" y="24" text-anchor="middle" font-size="14.5" font-weight="700" fill="#0f172a">Golden actor + snapshot resume</text><defs><marker id="g" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#16a34a"/></marker><marker id="d" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#d97706"/></marker><marker id="n" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#334155"/></marker><marker id="v" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#8b5cf6"/></marker></defs><rect x="280" y="64" width="160" height="54" rx="8" fill="#dcfce7" stroke="#16a34a" stroke-width="1.6"/><text x="360" y="86" text-anchor="middle" font-size="11" font-weight="700" fill="#14532d">golden actor</text><text x="360" y="102" text-anchor="middle" font-size="8" fill="#166534">memory snapshot on pause</text><rect x="500" y="66" width="200" height="50" rx="8" fill="#e2e8f0" stroke="#64748b" stroke-width="1.4"/><text x="600" y="86" text-anchor="middle" font-size="9.5" font-weight="700" fill="#334155">gs:// snapshot bucket</text><text x="600" y="101" text-anchor="middle" font-size="8" fill="#475569">cross-worker persistence</text><line x1="440" y1="86" x2="498" y2="86" stroke="#334155" stroke-width="1.7" stroke-dasharray="5 3" marker-end="url(#n)"/><text x="469" y="78" text-anchor="middle" font-size="7.5" fill="#475569">snapshot</text><rect x="160" y="150" width="240" height="50" rx="8" fill="#dbeafe" stroke="#60a5fa" stroke-width="1.5"/><text x="280" y="170" text-anchor="middle" font-size="10" font-weight="700" fill="#1e293b">new actor bind</text><text x="280" y="186" text-anchor="middle" font-size="8.5" fill="#475569">ResumeGoldenActor</text><line x1="360" y1="118" x2="300" y2="148" stroke="#16a34a" stroke-width="1.7" marker-end="url(#g)"/><text x="345" y="138" text-anchor="middle" font-size="7.5" fill="#166534">resume</text><rect x="430" y="150" width="270" height="50" rx="8" fill="#fef3c7" stroke="#d97706" stroke-width="1.4"/><text x="565" y="169" text-anchor="middle" font-size="9" font-weight="700" fill="#7c2d12">internal engine reconcile</text><text x="565" y="185" text-anchor="middle" font-size="8" fill="#92400e">no operator verb · automatic</text><text x="360" y="222" text-anchor="middle" font-size="10" fill="#64748b">Idle actor snapshots its memory, then resumes from a golden snapshot, not a cold boot. Automatic, no kubectl verb.</text></svg></div>

Pause, snapshot and resume are **automatic**: there is no kubectl verb, `ResumeGoldenActor` is an
internal phase of the substrate engine. The cell below shows the golden actor behind each
`SandboxAgent` and the snapshot it keeps on the worker. That snapshot is what a bind resumes from
instead of cold-booting a runtime, which is why the binds in 5.4 land in a few hundred milliseconds.

Resuming from a golden works with no configuration, and neither does persisting it. kagent already
points substrate at object storage and stamps every `ActorTemplate` with its own
`snapshotsConfig.location`, so snapshots survive a worker and an actor can come back somewhere else.
On this cluster the backend is the in-cluster RustFS bucket (`atelet` runs with an S3 backend, so the
`gs://` scheme in the location is nominal). To use your own bucket, set the location yourself:

```yaml
apiVersion: kagent.dev/v1alpha2
kind: SandboxAgent
metadata: { name: substrate-demo, namespace: kagent }
spec:
  substrate:
    workerPoolRef: { name: kagent-default }
    snapshotsConfig:
      location: gs://<your-bucket>/kagent/substrate-demo/
```

In [ ]:
templates substrate-demo                                   # the golden actor behind the agent, by UUID
kubectl --context $CTX -n kagent get actortemplate -o jsonpath='{range .items[*]}  {.metadata.name}{"  ->  "}{.spec.snapshotsConfig.location}{"\n"}{end}'
kubectl --context $CTX -n kagent get ds kagent-atelet -o jsonpath='{range .spec.template.spec.containers[0].env[*]}{.name}={.value}{"\n"}{end}' | grep -E 'ATE_STORAGE_BACKEND|AWS_ENDPOINT_URL' | sed 's/^/  atelet: /'


## 5.6 · Tools from inside the sandbox

A `SandboxAgent` carries the same `tools` block as an `Agent`, so a sandboxed actor can call MCP tools. Here the agent gets three read-only Kubernetes tools from kagent's tool server and is asked about the cluster it runs on. The tool call and the tool result come back in the A2A history as data parts, and kagent stores the task, so the trace of what the actor did is kept outside the sandbox even though the actor itself is checkpointed the moment it finishes.


In [ ]:
show $SUBSTRATE_YAML/sb-tools.yaml
kubectl --context $CTX apply -f $SUBSTRATE_YAML/sb-tools.yaml
wait-ready sandboxagent sb-tools $SUBSTRATE_YAML/sb-tools.yaml
SID=$(session sb-tools chat)                       # one session, kept for the next cell
ask-in $SID sb-tools "How many pods are in the kagent namespace, and how many are Running? Use k8s_get_resources. One sentence."


### The same conversation, a second turn, then delete it

One session is one actor for the whole life of the conversation. The second turn below resumes the actor from **its own** snapshot (the version that 5.2 showed going up), so it still has the first turn in memory and can answer from it. Then the session is deleted through kagent's sessions API and the actor goes with it: kagent owns the mapping from conversation to actor, so cleaning up a chat cleans up its sandbox.


In [ ]:
ask-in $SID sb-tools "Which of those pods has restarted, if any? Answer from what you already fetched. One sentence."
delete-session $SID


## 5.7 · A second pool for a different trust tier

Worker pools are the placement boundary. A `WorkerPool` carries a pod template (node selector, tolerations, resources), so a pool can be pinned to a node group with its own runtime, network policy or hardware. A `SandboxAgent` chooses its pool with `substrate.workerPoolRef`, and kagent turns that into the template's `workerSelector`, so the actor can never be placed anywhere else. Below: a second pool pinned to one node, an agent that only runs there, and the inventory showing three workers across two pools.


In [ ]:
show $SUBSTRATE_YAML/kagent-tier2.yaml
kubectl --context $CTX apply -f $SUBSTRATE_YAML/kagent-tier2.yaml
scale-pool 1 kagent-tier2
kubectl --context $CTX -n kagent get workerpool


In [ ]:
kubectl --context $CTX apply -f $SUBSTRATE_YAML/sb-tier2.yaml        # substrate.workerPoolRef: kagent-tier2
wait-ready sandboxagent sb-tier2 $SUBSTRATE_YAML/sb-tier2.yaml
templates sb-tier2                                          # the template can only select tier2 workers
ask sb-tier2 "Which worker pool do you run in? One sentence."
workers


## 5.8 · Change the agent's shape: a second golden beside the first

A golden snapshot is a memory image, so it cannot be edited in place. kagent puts a hash of the rendered actor spec into the `ActorTemplate` name, and what happens on a change depends on what changed. The system message and the tool list travel in the agent's Secret, which is outside the hash: change the prompt and new sessions pick it up on the same golden within seconds. Change the actor's **shape** (the image, the runtime, the worker pool, the snapshot location) and kagent renders a **second** template, builds its golden snapshot while the first keeps serving, and flips new sessions over once it is Ready. Sessions already open stay on the template they were born on, and the old template is kept, so moving back is a pointer move.

Below: the prompt change first (same template), then `sb-tools` moved onto the tier2 pool from 5.7 (a new template with a tier2 selector), a session opened before the move asked again, and the move reversed.


In [ ]:
templates sb-tools
kubectl --context $CTX -n kagent patch sandboxagent sb-tools --type=merge -p '{"spec":{"declarative":{"systemMessage":"You are an SRE assistant running inside a gVisor sandbox. Use your tools. Begin every reply with the word ROLLOUT."}}}'
sleep 5; templates sb-tools                                 # same template: a prompt is configuration
OLD=$(session sb-tools before-the-move); ask-in $OLD sb-tools "Reply with the single word ready."


In [ ]:
kubectl --context $CTX -n kagent patch sandboxagent sb-tools --type=merge -p '{"spec":{"substrate":{"workerPoolRef":{"name":"kagent-tier2"}}}}'
wait-templates sb-tools 2                                   # a second template and golden, built beside the first
templates sb-tools
ask sb-tools "Reply with the single word ready."            # a new session lands on the new template
ask-in $OLD sb-tools "Again, the single word ready."        # the old session stays on the one it was born on
sleep 8; actors sb-tools


In [ ]:
kubectl --context $CTX -n kagent patch sandboxagent sb-tools --type=merge -p '{"spec":{"substrate":{"workerPoolRef":{"name":"kagent-default"}}}}'
sleep 8; templates sb-tools                                 # moving back reuses the first template


## 5.9 · Pod agents and sandboxed agents in one graph

Nothing forces a whole system onto substrate. A `SandboxAgent` and a pod-backed `Agent` are both A2A servers in kagent, so either can list the other under `tools` with `type: Agent`. Below, an ordinary pod agent plans and delegates the cluster question to the sandboxed `sb-tools` actor: the part that holds tools and touches the cluster runs in gVisor and is checkpointed between calls, the coordinator stays a plain Deployment.


In [ ]:
show $SUBSTRATE_YAML/pod-planner.yaml
kubectl --context $CTX apply -f $SUBSTRATE_YAML/pod-planner.yaml
wait-pod-agent pod-planner
ask-pod pod-planner "How many pods are Running in the kagent namespace? Delegate to sb-tools and relay the number."


## 5.10 · The worker fleet

A `WorkerPool` is a fleet of pre-warmed worker pods, and every agent runs as a gVisor actor packed onto one of them. Adding an agent binds a new actor onto a warm worker in about a second; it does not cost a pod. The fleet scales on its own, so you add workers for capacity without touching the agents.

<div align="center"><svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 760 300" style="width:100%;max-width:1000px;height:auto" font-family="-apple-system,Segoe UI,Roboto,sans-serif"><rect x="0" y="0" width="760" height="300" rx="10" fill="#f8fafc"/><text x="380" y="26" text-anchor="middle" font-size="15" font-weight="700" fill="#0f172a">Agent Substrate: one WorkerPool, many gVisor actors</text><text x="380" y="44" text-anchor="middle" font-size="10.5" fill="#475569">each SandboxAgent runs as a gVisor actor; actors pack onto shared, pre-warmed worker pods</text><rect x="20" y="58" width="720" height="192" rx="10" fill="#eff6ff" stroke="#3b82f6" stroke-width="1.6"/><text x="36" y="78" font-size="11" font-weight="700" fill="#1e40af">WorkerPool · kagent-default · sandboxClass gvisor</text><rect x="36" y="90" width="216" height="148" rx="8" fill="#e2e8f0" stroke="#64748b" stroke-width="1.4"/><text x="144" y="110" text-anchor="middle" font-size="11" font-weight="700" fill="#334155">worker pod 1</text><rect x="54" y="120" width="180" height="46" rx="6" fill="#dcfce7" stroke="#16a34a" stroke-width="1.3"/><text x="144" y="139" text-anchor="middle" font-size="10" font-weight="700" fill="#14532d">actor · checkout</text><text x="144" y="154" text-anchor="middle" font-size="8" fill="#166534">gVisor (runsc) sandbox</text><rect x="54" y="176" width="180" height="46" rx="6" fill="#dcfce7" stroke="#16a34a" stroke-width="1.3"/><text x="144" y="195" text-anchor="middle" font-size="10" font-weight="700" fill="#14532d">actor · search</text><text x="144" y="210" text-anchor="middle" font-size="8" fill="#166534">gVisor (runsc) sandbox</text><rect x="272" y="90" width="216" height="148" rx="8" fill="#e2e8f0" stroke="#64748b" stroke-width="1.4"/><text x="380" y="110" text-anchor="middle" font-size="11" font-weight="700" fill="#334155">worker pod 2</text><rect x="290" y="120" width="180" height="46" rx="6" fill="#dcfce7" stroke="#16a34a" stroke-width="1.3"/><text x="380" y="139" text-anchor="middle" font-size="10" font-weight="700" fill="#14532d">actor · pricing</text><text x="380" y="154" text-anchor="middle" font-size="8" fill="#166534">gVisor (runsc) sandbox</text><rect x="290" y="176" width="180" height="46" rx="6" fill="#dcfce7" stroke="#16a34a" stroke-width="1.3"/><text x="380" y="195" text-anchor="middle" font-size="10" font-weight="700" fill="#14532d">actor · support</text><text x="380" y="210" text-anchor="middle" font-size="8" fill="#166534">gVisor (runsc) sandbox</text><rect x="508" y="90" width="216" height="148" rx="8" fill="#e2e8f0" stroke="#64748b" stroke-width="1.4"/><text x="616" y="110" text-anchor="middle" font-size="11" font-weight="700" fill="#334155">worker pod 3</text><rect x="526" y="120" width="180" height="46" rx="6" fill="#dcfce7" stroke="#16a34a" stroke-width="1.3"/><text x="616" y="139" text-anchor="middle" font-size="10" font-weight="700" fill="#14532d">actor · summarizer</text><text x="616" y="154" text-anchor="middle" font-size="8" fill="#166534">gVisor (runsc) sandbox</text><rect x="526" y="176" width="180" height="46" rx="6" fill="#dcfce7" stroke="#16a34a" stroke-width="1.3"/><text x="616" y="195" text-anchor="middle" font-size="10" font-weight="700" fill="#14532d">actor · planner</text><text x="616" y="210" text-anchor="middle" font-size="8" fill="#166534">gVisor (runsc) sandbox</text><text x="380" y="278" text-anchor="middle" font-size="10.5" fill="#334155">Add an agent and a new actor binds onto a warm worker in about a second, not a new pod. Scale the pool to add workers.</text></svg></div>

Below: the live fleet and an elastic resize (2 to 4 workers and back, with the actors untouched).

In [ ]:
kubectl --context $CTX -n kagent get pods -l ate.dev/worker-pool=kagent-default -o wide
scale-pool 4                                                # 2 -> 4 workers, actors untouched
kubectl --context $CTX -n kagent get actortemplate --no-headers | wc -l | tr -d ' ' | sed 's/^/  gVisor actors still: /'
scale-pool 2                                                # workers are cattle, the actors carried on


## 5.11 · Chat with the sandboxed agent

The proof that matters: talk to it. A SandboxAgent is an A2A server. Regular agents answer on `/api/a2a/<ns>/<name>/`; sandboxed ones answer on `/api/a2a-sandboxes/<ns>/<name>/`. Every message carries a `contextId`, which is a kagent session id, so we open a session and then send the prompt. The reply comes straight from the ADK agent running inside the gVisor actor.

In [ ]:
ask substrate-demo "In one sentence, what is 17 times 3?"
ask substrate-demo "Name one benefit of running an agent in a gVisor sandbox. One sentence."


## 5.12 · Watch it live: Substrate Scope

`kubectl get pods` makes substrate look boring, because the interesting state is not in pods. [Substrate Scope](https://github.com/themsquared/substrate-scope) (Mike Moore, Apache-2.0, not a Solo product) draws the runtime the way it works: worker bays across the top with the actor each one is running, the restore queue when demand exceeds the pool, every suspended actor on the object-storage shelf with its snapshot count, and reserved capacity plotted against what the same agents would reserve as always-on pods. It reads kagent's inventory and sessions APIs through a port-forward and scales the pool with `kubectl scale` when you press the buttons.

The cell below starts the viewer, deploys a handful of load agents and sends real chats at them, so the board fills. These are billable model calls; the generator stops at the budget, and `substrate-load.sh stop` removes the agents and the viewer.


In [ ]:
AGENTS=4 CHATS=20 SUBSTRATE_CTX=$CTX ./demo-scripts/substrate-load.sh     # then open http://localhost:8123


---

## Setup

Everything below runs **before** the demo, so none of it is on screen while you present. In order: Connect, then Enable substrate once per cluster.


### Connect

Run this first, every time. It puts the demo's helpers in the shell (`ask`, `actors`, `workers`, `watch-turn`, `catch-runsc`, `on-node`, `fire`, `scale-pool`, `templates`, `delete-session`) and points them at the `kind-substrate` cluster. It changes nothing in the cluster, so it is safe to re-run at any point, including halfway through the demo. The same line works pasted into a terminal, or use `source demo-scripts/env.sh 5`.


In [ ]:
# whether the kernel started in this folder or at the repo root, one of these is it
source demo-scripts/substrate-lib.sh 2>/dev/null ||
source vision-demo-2026/demo-scripts/substrate-lib.sh


### Enable substrate (once per cluster)

Substrate is **off by default**. This creates the `kind-substrate` cluster if it is missing, installs kagent-enterprise with the substrate control plane and a two-worker gVisor `WorkerPool`, and proves an actor can be placed before it reports done. Idempotent; a few minutes the first time. On Apple Silicon it uses the arm64 worker image.


In [ ]:
./demo-scripts/substrate-cluster.sh


---

## Teardown

Removes every object any cell above can create, puts the pool back to two workers and stops the viewer. Substrate itself stays installed; the comment in the cell removes it too.


In [ ]:
./demo-scripts/substrate-load.sh stop >/dev/null 2>&1 || true
kubectl --context $CTX -n kagent delete sandboxagent substrate-demo substrate-demo-2 substrate-demo-3 substrate-demo-4 sb-tools sb-tier2 --ignore-not-found
kubectl --context $CTX -n kagent delete agent pod-baseline-1 pod-baseline-2 pod-baseline-3 pod-planner --ignore-not-found
kubectl --context $CTX -n kagent delete workerpool kagent-tier2 --ignore-not-found
kubectl --context $CTX -n kagent scale workerpool/kagent-default --replicas=2 >/dev/null 2>&1
pf-down; echo "✓ demo objects removed; the pool is back to 2 workers"
# to remove substrate itself:
#   helm --kube-context $CTX upgrade kagent "$KENT_CHART" -n kagent --reuse-values \
#     --set substrate.enabled=false --set substrateWorkerPool.create=false --set controller.substrate.enabled=false
#   kubectl --context $CTX -n kagent delete workerpool kagent-default --ignore-not-found
